In [0]:
%pip install gspread google-auth
dbutils.library.restartPython()

In [0]:
# Date/time range
dbutils.widgets.text("from_date", "17/07/2026", "From Date (dd/MM/yyyy)")
dbutils.widgets.text("from_time", "17:30", "From Time (HH:mm)")
dbutils.widgets.text("to_date", "17/07/2026", "To Date (dd/MM/yyyy)")
dbutils.widgets.text("to_time", "19:15", "To Time (HH:mm)")

REPORT_NAMES = [
    "D.Analysis - OSR PiE",
    "D.Analysis - OSR Topup",
    "D.Analysis - E3 Packing",
    "D.Analysis - Parcel Induct",
    "D.Analysis - Parcel Sortation",
    "D.Analysis - Inbound Decanting",
    "D.Analysis - OSR Decanting",
    "D.Analysis - BCR Inducting",
    "D.Analysis - E1/E2 Inducting",
    "D.Analysis - Sorter 6 Packing",
    "D.Analysis - Online Picking - Drive",
    "D.Analysis - Online Picking - Way",
    "D.Analysis - Online Picking - E3",
    "D.Analysis - E3 BPP",
    "D.Analysis - E1/E2 BPP",
    "D.Analysis - RSPS Top Up",
    "D.Analysis - RSPS Pick",
    "D.Analysis - ISPS Top Up",
    "D.Analysis - ISPS Pick",
]

# Filter mode + multi-select of report names
dbutils.widgets.dropdown("filter_mode", "All", ["All", "Include selected", "Exclude selected"], "Filter Mode")
dbutils.widgets.multiselect("selected_reports", REPORT_NAMES[0], REPORT_NAMES, "Select Reports")

# Where to send the backfill result. Default is a table shown in Databricks
# (nothing written to the sheet); choose "Google Sheet" to append to the Data
# tab and rebuild Processed Data (15mins), like the live job.
dbutils.widgets.dropdown("output_target", "Databricks table", ["Databricks table", "Google Sheet"], "Output Target")

from_date = dbutils.widgets.get("from_date")
from_time = dbutils.widgets.get("from_time")
to_date   = dbutils.widgets.get("to_date")
to_time   = dbutils.widgets.get("to_time")
filter_mode = dbutils.widgets.get("filter_mode")
selected = [s.strip() for s in dbutils.widgets.get("selected_reports").split(",") if s.strip()]
output_target = dbutils.widgets.get("output_target")

print(f"Backfill range: {from_date} {from_time} -> {to_date} {to_time} (UK local time)")
print(f"Filter: {filter_mode}" + (f" -> {selected}" if filter_mode != "All" else ""))
print(f"Output target: {output_target}")


In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

uk = ZoneInfo("Europe/London")

start_local = datetime.strptime(f"{from_date} {from_time}", "%d/%m/%Y %H:%M").replace(tzinfo=uk)
end_local   = datetime.strptime(f"{to_date} {to_time}",   "%d/%m/%Y %H:%M").replace(tzinfo=uk)

# Validate alignment to 15-min boundaries
for label, t in [("From", start_local), ("To", end_local)]:
    if t.minute % 15 != 0:
        raise ValueError(f"{label} time {t.strftime('%H:%M')} is not on a 15-minute boundary (:00, :15, :30, :45)")

if end_local <= start_local:
    raise ValueError("To datetime must be after From datetime")

# Build each 15-min block as (start, end) pairs in UK local, plus UTC equivalents for filtering
blocks = []
cursor = start_local
while cursor < end_local:
    block_start_local = cursor
    block_end_local   = cursor + timedelta(minutes=15)
    blocks.append({
        "start_local": block_start_local,
        "end_local":   block_end_local,
        "start_utc":   block_start_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S"),
        "end_utc":     block_end_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S"),
    })
    cursor = block_end_local

print(f"{len(blocks)} blocks to backfill:")
for b in blocks:
    print(f"  {b['start_local'].strftime('%d/%m/%Y %H:%M')} - {b['end_local'].strftime('%d/%m/%Y %H:%M')}")

In [0]:
import gspread
from google.oauth2.service_account import Credentials
import json

KEY_FILE_PATH = "/Workspace/Users/pakhei_tsang@next.co.uk/advance-mantis-398714-2168c9162641.json"  # adjust to your path

with open(KEY_FILE_PATH, "r") as f:
    creds_dict = json.load(f)

creds = Credentials.from_service_account_info(
    creds_dict, scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)

SHEET_ID = "1uiv27J0YPDWqSVuN-QLpS-fOcfxAjewK5ajTd3aA8vI"
sh = gc.open_by_key(SHEET_ID)
ws = sh.worksheet("Data")

print("Connected to:", sh.title)

In [0]:
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))
df.createOrReplaceTempView("landing_bonus_hub_event_parsed")

# Online Picking carries its zone inside a location string rather than in an
# attribute of its own: "location:XH117B" decodes as zone X, aisle H, bay 117,
# level B. So a zone filter is a test on the FIRST CHARACTER after the key, not
# a match on a whole attribute value.
#
# Matched case-insensitively, and the prefix is stripped by cutting at the
# first colon rather than by a fixed length. Both matter: the real values are
# lower case with no space after the colon, so 'Location: %' matched none of
# them and these three reports returned nothing at all rather than too little.
def _zone_pred(*zones):
    _z = ", ".join(f"'{z}'" for z in zones)
    return ("AND EXISTS(PAYLOAD_ATTRIBUTES, x -> lower(x) LIKE 'location:%' AND "
            "upper(substring(trim(regexp_replace(x, '^[^:]*:', '')), 1, 1)) "
            f"IN ({_z}))")


all_reports = [
    # attributes: this report's PAYLOAD_ATTRIBUTES reach the Data tab's
    # Attribute column, and its rows split by attribute set as a result.
    {"name": "D.Analysis - OSR PiE",
     "area": "('Pick','Pie Station')", "event": "('COUN','MSKU','PSKU','RSIN')", "extra": "",
     "attributes": True},
    {"name": "D.Analysis - OSR Topup",
     "area": "('Topup','TopUp','PiOrQi')", "event": "('QSIN','TARS','TPUT')", "extra": ""},
    {"name": "D.Analysis - E3 Packing",
     "area": "('E3 - Packing')", "event": "('PackingParcelCompleteEvent','PackingItemScannedEvent')", "extra": ""},
    {"name": "D.Analysis - Parcel Induct",
     "area": "('Parcel Induct')", "event": "('APAR','PIND','SPAR','UPAR')", "extra": ""},
    {"name": "D.Analysis - Parcel Sortation",
     "area": None, "event": "('ParcelSortedToSack','SackMappedToPosition','SackUnMappedFromPosition')", "extra": ""},
    {"name": "D.Analysis - Inbound Decanting",
     "area": "('Inbound Decanting')", "event": "('DECN','TOPR')", "extra": ""},
    {"name": "D.Analysis - OSR Decanting",
     "area": "('OSR Decanting')", "event": "('ODEC','OTOP')", "extra": ""},
    {"name": "D.Analysis - BCR Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%RET%')"},
    {"name": "D.Analysis - E1/E2 Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%PIE4EDW%')"},

    # No event filter: every event type in this area earns standard hours.
    # Only PackingItemScannedEvent counts towards VOLUME - a separate concern,
    # handled by vol_events in PROC_AREAS.
    {"name": "D.Analysis - Sorter 6 Packing",
     "area": "('Sorter 6 - Packing')", "event": None, "extra": ""},

    # Online Picking is ONE source split three ways by zone: same warehouse,
    # same area code, different zones decoded out of the location string. No
    # event filter, so every event type earns standard hours; only PickItemEvent
    # counts towards volume (see vol_events).
    #
    # attributes: "zone_aisle" puts "Zone B, Aisle A" on the Data tab rather
    # than the raw location. Grouping follows the Attribute column, so these
    # reports collapse to one row per operator per event type per window per
    # zone-and-aisle. Carrying the full location instead would group by bay and
    # level too - a row for every item picked."
    #
    # A row somehow carrying two locations in different zones would feed both
    # reports; the array + explode tagging below keeps that working rather than
    # silently picking one.
    {"name": "D.Analysis - Online Picking - Drive",
     "area": "('Online Picking')", "event": None,
     "extra": _zone_pred('D', 'F', 'Q', 'S', 'T', 'V'),
     "attributes": "zone_aisle"},
    {"name": "D.Analysis - Online Picking - Way",
     "area": "('Online Picking')", "event": None,
     "extra": _zone_pred('A', 'B', 'E', 'G', 'J', 'L', 'X', 'W', 'R'),
     "attributes": "zone_aisle"},
    {"name": "D.Analysis - Online Picking - E3",
     "area": "('Online Picking')", "event": None,
     "extra": _zone_pred('H', 'C'),
     "attributes": "zone_aisle"},
    # The two BPP areas are plain area filters with no event filter, so every
    # event type earns standard hours and only the one scan event below counts
    # towards volume. attributes: True puts the raw attribute list in the
    # Attribute column, the same treatment OSR PiE gets.
    {"name": "D.Analysis - E3 BPP",
     "area": "('E3 BPP Packing')", "event": None, "extra": "",
     "attributes": True},
    {"name": "D.Analysis - E1/E2 BPP",
     "area": "('BPP')", "event": None, "extra": "",
     "attributes": True},

    # Returns processing. RSPS and ISPS share event types (TPUT, RPUT) and are
    # told apart by area alone, so each pair needs BOTH filters - an event
    # filter on its own would merge them.
    {"name": "D.Analysis - RSPS Top Up",
     "area": "('ReturnsBuffer')", "event": "('TPUT','TART')", "extra": ""},
    {"name": "D.Analysis - RSPS Pick",
     "area": "('ReturnsBuffer')", "event": "('RPUT','RSEX')", "extra": ""},
    {"name": "D.Analysis - ISPS Top Up",
     "area": "('RSPS2Messaging')", "event": "('TPUT')", "extra": ""},
    {"name": "D.Analysis - ISPS Pick",
     "area": "('RSPS2Messaging')", "event": "('RPUT')", "extra": ""},
]

if filter_mode == "Include selected":
    reports = [r for r in all_reports if r['name'] in selected]
elif filter_mode == "Exclude selected":
    reports = [r for r in all_reports if r['name'] not in selected]
else:  # All
    reports = all_reports

if not reports:
    raise ValueError("Your filter selection matched no reports — check Filter Mode and Selected Reports")

print(f"Reports to run ({len(reports)} of {len(all_reports)}):")
for r in reports:
    print(f"  - {r['name']}")

COLUMNS = ['Date', 'Hour', 'PAYLOAD_BONUSCODE', 'PAYLOAD_EVENTTYPE', 'Attribute',
           'Total_Quantity', 'Total_StandardHours', 'Total_SMV',
           'Week', 'Date Time Range', 'Report Name']

In [0]:
# --- Std Hours divisors, read from row 2 of Front. C2 is the site-wide value
#     (the same one the Apps Script side used before processing moved into
#     Databricks). D2 is Sorter 6 Packing's own, because its standard hours are
#     set on a different basis; which areas use which is declared in PROC_AREAS
#     below rather than hard-coded here. Both cells accept a percentage
#     ("96.40%") or a bare number, and one read covers both. `sh` is the
#     spreadsheet handle from the connection cell above. ---
DIVISOR_DEFAULT_CELL = "C2"

_div_cells = sh.worksheet("Front").get("C2:D2") or []
_div_row = _div_cells[0] if _div_cells else []


def _parse_divisor(raw, cell):
    """A divisor cell -> float, or None when the cell is empty."""
    raw = (raw or "").strip()
    if not raw:
        return None
    try:
        value = float(raw.rstrip('%')) / 100 if raw.endswith('%') else float(raw)
    except ValueError:
        raise ValueError(f"Front!{cell} is not a number or a percentage: {raw!r}")
    # Zero would divide every standard hour into infinity. 1.0 leaves the hours
    # exactly as they arrived, which is the honest reading of "no divisor set".
    return 1.0 if value == 0 else value


divisor = _parse_divisor(_div_row[0] if len(_div_row) > 0 else "", "C2")
if divisor is None:
    raise ValueError("Front!C2 is empty - cannot determine the Std Hours divisor.")

_sorter6_divisor = _parse_divisor(_div_row[1] if len(_div_row) > 1 else "", "D2")

# An empty D2 falls back to the site-wide value rather than failing the run: a
# slightly wrong figure for one area beats no data for any of them. It is
# printed either way, so "D2 was never filled in" is visible in the run log
# rather than something to deduce from the numbers.
_divisor_by_cell = {"C2": divisor}
if _sorter6_divisor is not None:
    _divisor_by_cell["D2"] = _sorter6_divisor

print(f"Divisor (Front!C2, site-wide): {divisor}")
print("Divisor (Front!D2, Sorter 6 Packing): " +
      (f"{_sorter6_divisor}" if _sorter6_divisor is not None
       else f"EMPTY - falling back to Front!C2 ({divisor})"))

# --- Work-area pivot config: the column ORDER of Processed Data (15mins), each
#     area's volume event type(s), and the short label its volume column carries.
#     Single source of truth - the header row, the data rows and the clear range
#     are all derived from this list, so adding an area here is the only edit
#     the tab's layout needs. ---
PROC_AREAS = [
    {"report": "D.Analysis - OSR PiE",           "label": "PiE",               "vol_events": {"MSKU", "PSKU"}, "divisor_cell": "C2"},
    {"report": "D.Analysis - OSR Topup",         "label": "Top Up",            "vol_events": {"TPUT"}, "divisor_cell": "C2"},
    {"report": "D.Analysis - E3 Packing",        "label": "E3 Packing",        "vol_events": {"PackingItemScannedEvent"}, "divisor_cell": "C2"},
    {"report": "D.Analysis - Parcel Sortation",  "label": "Parcel Sortation",  "vol_events": {"ParcelSortedToSack"}, "divisor_cell": "C2"},
    {"report": "D.Analysis - Parcel Induct",     "label": "Parcel Induct",     "vol_events": {"SPAR"}, "divisor_cell": "C2"},
    {"report": "D.Analysis - Inbound Decanting", "label": "Inbound Decanting", "vol_events": {"DECN"}, "divisor_cell": "C2"},
    {"report": "D.Analysis - OSR Decanting",     "label": "OSR Decanting",     "vol_events": {"ODEC"}, "divisor_cell": "C2"},
    {"report": "D.Analysis - BCR Inducting",     "label": "BCR Inducting",     "vol_events": {"SPOS"}, "divisor_cell": "C2"},
    {"report": "D.Analysis - E1/E2 Inducting",   "label": "E1/E2 Inducting",   "vol_events": {"SPOS"}, "divisor_cell": "C2"},
    # divisor_cell: Sorter 6 Packing divides its standard hours by Front!D2
    # rather than the site-wide C2. Any area can name its own cell this way.
    {"report": "D.Analysis - Sorter 6 Packing",  "label": "Sorter 6 Packing",  "vol_events": {"PackingItemScannedEvent"}, "divisor_cell": "D2"},
    {"report": "D.Analysis - Online Picking - Drive", "label": "Drive - Online Picking", "vol_events": {"PickItemEvent"}, "divisor_cell": None},
    {"report": "D.Analysis - Online Picking - Way",   "label": "Way - Online Picking",   "vol_events": {"PickItemEvent"}, "divisor_cell": None},
    {"report": "D.Analysis - Online Picking - E3",    "label": "E3 - Online Picking",    "vol_events": {"PickItemEvent"}, "divisor_cell": None},
    {"report": "D.Analysis - E3 BPP",                "label": "E3 BPP",                "vol_events": {"Scan Item to Carton"}, "divisor_cell": None},
    {"report": "D.Analysis - E1/E2 BPP",             "label": "E1/E2 BPP",             "vol_events": {"BppScanItemToCarton"}, "divisor_cell": None},
    {"report": "D.Analysis - RSPS Top Up",         "label": "RSPS Top Up",         "vol_events": {"TPUT"}, "divisor_cell": None},
    {"report": "D.Analysis - RSPS Pick",           "label": "RSPS Pick",           "vol_events": {"RPUT"}, "divisor_cell": None},
    {"report": "D.Analysis - ISPS Top Up",         "label": "ISPS Top Up",         "vol_events": {"TPUT"}, "divisor_cell": None},
    {"report": "D.Analysis - ISPS Pick",           "label": "ISPS Pick",           "vol_events": {"RPUT"}, "divisor_cell": None},
]

# The header row is written from PROC_AREAS as well, rather than maintained by
# hand on the tab. The dashboard locates every column by its header NAME, so a
# header that disagrees with the data underneath it is the one mistake that
# mis-maps the whole tab silently - and inserting a work area shifts every
# column after it. Deriving both from this list makes that disagreement
# impossible.
PROC_HEADER = (["Date", "Hour", "BONUS"]
               + [a["report"] for a in PROC_AREAS]
               + ["Sum of Std hrs"]
               + [f"Volume - {a['label']}" for a in PROC_AREAS])


def _col_a1(n):
    """1-based column number -> A1 letters (24 -> 'X')."""
    s = ""
    while n:
        n, r = divmod(n - 1, 26)
        s = chr(65 + r) + s
    return s


PROC_LAST_COL = _col_a1(len(PROC_HEADER))

# Per-area divisor, in PROC_AREAS order so the pivot can index it alongside the
# standard-hours totals.
#
# "divisor_cell" is one of three things:
#   "C2"/"D2"  divide by that cell
#   None       do NOT divide - the hours are used exactly as they arrive
#   absent     the site-wide default, kept only so an area added without
#              thinking about it behaves as everything used to
#
# None is spelled out rather than left to a default because "no adjustment" is
# a decision about that area, and a reader should not have to infer it from a
# missing key.
def _divisor_for(area):
    cell = area["divisor_cell"] if "divisor_cell" in area else DIVISOR_DEFAULT_CELL
    if cell is None:
        return 1.0
    got = _divisor_by_cell.get(cell)
    if got is None:
        # Named a cell that turned out to be empty. Falling back to the
        # site-wide value keeps the run alive; saying so keeps it from being
        # mistaken for a deliberate setting.
        print(f"  WARNING: {area['report']} names Front!{cell}, which is empty - "
              f"using Front!{DIVISOR_DEFAULT_CELL} ({divisor})")
        return divisor
    return got


PROC_DIVISORS = [_divisor_for(a) for a in PROC_AREAS]

print("Std Hours divisor by area:")
for _a, _d in zip(PROC_AREAS, PROC_DIVISORS):
    _cell = _a["divisor_cell"] if "divisor_cell" in _a else DIVISOR_DEFAULT_CELL
    print(f"  {_a['label']:<24} {'no divisor' if _cell is None else 'Front!' + _cell:<12} /{_d}")


In [0]:
import pandas as pd
from datetime import datetime, timezone, date

# ============================================================
# CONSOLIDATED FETCH + OUTPUT
# One Delta scan for the WHOLE backfill (all blocks x selected reports),
# instead of len(blocks) x len(reports) separate spark.sql().toPandas() calls.
#
# Respects Filter Mode / Select Reports: the CASE/predicate array is built from
# `reports` (already filtered in the config cell), so only the selected reports
# are scanned, tagged and output. Using a SET (array + explode) rather than a
# first-match CASE preserves the original per-report semantics exactly - a row
# that satisfied two reports' filters fed both, and still does.
#
# The scan is bounded to [min start, max end] of the blocks for partition
# pruning, then filtered to the EXACT 15-min buckets (window_epoch IN ...).
#
# Output goes where the output_target widget says: "Google Sheet" appends to the
# Data tab in one batch (and the next cell rebuilds Processed Data); the default
# "Databricks table" writes nothing to the sheet and just displays the result.
# ============================================================

EPOCH_REF = date(2025, 12, 14)  # same week-numbering anchor as the live job
def _week_val(d):
    return ((d - EPOCH_REF).days // 7 + 46) % 52

def _blk_epoch(s):
    return int(datetime.strptime(s, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc).timestamp())

# 15-min bucket (UTC epoch) -> block metadata
blk_by_epoch = {}
for b in blocks:
    blk_by_epoch[_blk_epoch(b['start_utc'])] = {
        "week_val": _week_val(b['start_local'].date()),
        "date_time_range": f"{b['start_local'].strftime('%d/%m/%Y %H:%M')} - {b['end_local'].strftime('%d/%m/%Y %H:%M')}",
        "date_display": b['start_local'].strftime('%d/%m/%Y'),
    }

overall_start_utc = min(b['start_utc'] for b in blocks)
overall_end_utc = max(b['end_utc'] for b in blocks)
epoch_list = ", ".join(str(e) for e in sorted(blk_by_epoch))

# One report's {area, event, extra} config -> a standalone SQL boolean predicate
# (the `extra` field starts with "AND ", stripped so it can stand alone).
def _report_predicate(r):
    parts = []
    if r['area']:
        parts.append(f"PAYLOAD_AREACODE IN {r['area']}")
    # event is optional in the same way area already is: Sorter 6 Packing
    # earns standard hours from EVERY event type in its area, so it carries
    # no event filter. Volume stays restricted, via PROC_AREAS' vol_events.
    if r['event']:
        parts.append(f"PAYLOAD_EVENTTYPE IN {r['event']}")
    if not parts:
        raise ValueError(
            f"Report {r['name']!r} has neither an area nor an event filter - "
            "it would match every event in the warehouse."
        )
    extra = r['extra'].strip()
    if extra:
        if extra[:4].upper() == "AND ":
            extra = extra[4:].strip()
        parts.append(extra)
    return " AND ".join(f"({p})" for p in parts)

preds = [(_report_predicate(r), r['name']) for r in reports]
case_exprs = ",\n            ".join(f"CASE WHEN {pred} THEN '{name}' END" for pred, name in preds)
match_any = " OR ".join(f"({pred})" for pred, _ in preds)

# An Online Picking event describes where it happened as "location:XH117B" -
# zone X, aisle H, bay 117, level B - and a ChangeAisleEvent also carries
# "lastLocation:VK162F" for where it came from.
#
# Only zone and aisle are kept. Bay and level would group an operator's
# picks down one aisle into a row each; zone and aisle collapse them to one.
def _attr_sql(key):
    """First attribute with this key, or NULL.

    Case-insensitive: the real values are lower case with no space after the
    colon, so a pattern of 'Location: %' matches none of them.
    "lastLocation:..." does not start with "location:", so a prefix test
    cannot confuse the two keys.

    try_element_at, NOT element_at: under ANSI mode element_at raises
    INVALID_ARRAY_INDEX_IN_ELEMENT_AT on an EMPTY array rather than
    returning NULL, and an event with no such attribute filters down to
    exactly that.
    """
    return ("try_element_at(filter(PAYLOAD_ATTRIBUTES, "
            f"x -> lower(x) LIKE '{key.lower()}:%'), 1)")

def _zone_aisle_sql(attr):
    """'location:XH117B' -> 'Zone X, Aisle H'.

    The key is cut at the first colon rather than by a fixed length, so it
    serves location and lastLocation alike. Anything too short to hold both
    characters, or absent entirely, gives '' rather than a half-formed label.
    """
    c = f"trim(regexp_replace({attr}, '^[^:]*:', ''))"
    return (f"CASE WHEN {c} IS NULL OR length({c}) < 2 THEN '' "
            f"ELSE concat('Zone ', upper(substring({c}, 1, 1)), "
            f"', Aisle ', upper(substring({c}, 2, 1))) END")

_za = _zone_aisle_sql(_attr_sql("location"))
_za_last = _zone_aisle_sql(_attr_sql("lastLocation"))

# The last location is appended when the event carries one, keyed off the
# attribute being present rather than off the event being named
# ChangeAisleEvent - so any other event carrying one is handled too.
zone_aisle_sql = (f"CASE WHEN {_za} = '' THEN '' "
                  f"WHEN {_za_last} = '' THEN {_za} "
                  f"ELSE concat({_za}, '; Last location: ', {_za_last}) END")

# A report's `attributes` says WHAT its Attribute column holds: True for the
# raw attribute list, or the name of a column derived in the CTE below.
# Whichever it is, the value joins the GROUP BY for that report alone, so
# its rows split by that value while every other report gets '' and groups
# exactly as it did before.
_attr_cases = []
for _r in reports:
    _a = _r.get('attributes')
    if not _a:
        continue
    _expr = 'attrs' if _a is True else _a
    _attr_cases.append(f"WHEN report_name = '{_r['name']}' THEN {_expr}")
if _attr_cases:
    attr_sel = "CASE " + " ".join(_attr_cases) + " ELSE '' END"
    # Repeated in GROUP BY rather than referenced by alias, which not every
    # Spark version accepts.
    attr_group = ",\n        " + attr_sel
else:
    attr_sel = "''"
    attr_group = ""

query = f"""
  WITH base AS (
    SELECT
      date_format(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London'),'dd/MM/yyyy') AS Date,
      hour(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London')) AS Hour,
      CAST(FLOOR(unix_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP)) / 900) * 900 AS BIGINT) AS window_epoch,
      -- Canonicalised here, at the point the value enters the pipeline.
      -- The bonus code is an identifier and the source is not consistent
      -- about case, so "mf5" and "MF5" arrived as two different people: two
      -- rows out of this GROUP BY, two rows on the Data tab, two rows in the
      -- pivot, two heads on the dashboard, and each with half the standard
      -- hours they had actually earned. Folding case in the SQL merges them
      -- before anything downstream can tell them apart.
      upper(trim(PAYLOAD_BONUSCODE)) AS bonus,
      PAYLOAD_EVENTTYPE AS eventtype,
      -- Flattened here so the Data tab gets one readable cell rather than a
      -- stringified array. concat_ws drops nulls and yields '' for an empty
      -- or absent array, which is what an event with no attributes should
      -- show.
      concat_ws(', ', PAYLOAD_ATTRIBUTES) AS attrs,
      -- Zone and aisle, decoded from the location attribute. See zone_aisle_sql.
      {zone_aisle_sql} AS zone_aisle,
      PAYLOAD_QUANTITY      AS qty,
      PAYLOAD_STANDARDHOURS AS std,
      PAYLOAD_SMV           AS smv,
      filter(array(
        {case_exprs}
      ), x -> x IS NOT NULL) AS report_names
    FROM landing_bonus_hub_event_parsed
    WHERE TRIM(PAYLOAD_WAREHOUSECODE) = 'X'
      AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{overall_start_utc}')
      AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{overall_end_utc}')
      AND ( {match_any} )
  )
  SELECT
    Date, Hour, window_epoch, bonus, eventtype,
    {attr_sel} AS attribute,
    report_name,
    SUM(qty) AS Total_Quantity,
    SUM(std) AS Total_StandardHours,
    SUM(smv) AS Total_SMV
  FROM base
  LATERAL VIEW explode(report_names) t AS report_name
  WHERE window_epoch IN ({epoch_list})
  GROUP BY Date, Hour, window_epoch, bonus, eventtype, report_name{attr_group}
"""

report_order = {r['name']: i for i, r in enumerate(reports)}

failures = []
total_rows_pushed = 0
pdf = None
try:
    pdf = spark.sql(query).toPandas()
    print(f"Consolidated query returned {len(pdf)} grouped rows "
          f"across {len(blocks)} block(s) x {len(reports)} report(s).")
except Exception as e:
    error_msg = f"{type(e).__name__}: {str(e)}"
    print(f"CONSOLIDATED QUERY FAILED - {error_msg}")
    failures.append({"block": f"{overall_start_utc}..{overall_end_utc} UTC",
                     "report": "(consolidated query)", "error": error_msg})

# --- Shape into the Data-tab COLUMNS order; note which blocks had data. ---
#
# Values go to the sheet as their real types and are written RAW below.
# USER_ENTERED asks Sheets to INTERPRET each value, and column C is the bonus
# code: it turned "3E3" into the number 3000 (read back as "3.00E+03") and
# "2PM" into a time (read back as "14:00"), so one operator became two. The
# live job had the identical bug; this keeps the two writers producing
# byte-identical rows, which matters because they both feed the same tab.

# Sheets rejects NaN/Infinity outright - they are not valid JSON - and a blank
# cell is the honest representation of "no figure" anyway.
def _num(v):
    try:
        f = float(v)
    except (TypeError, ValueError):
        return ''
    if f != f or f in (float('inf'), float('-inf')):
        return ''
    return f

def _int(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return ''

data_rows = []
blocks_with_data = set()
if pdf is not None:
    for _, row in pdf.iterrows():
        ep = int(row['window_epoch'])
        m = blk_by_epoch.get(ep)
        if m is None:
            continue  # safety: outside target buckets
        blocks_with_data.add(ep)
        hour_key = int(row['Hour']) if str(row['Hour']).strip().lstrip('-').isdigit() else 0
        data_rows.append({
            "sort": (ep, report_order.get(row['report_name'], 999), hour_key, str(row['bonus'])),
            "vals": [str(row['Date']),
                     _int(row['Hour']),
                     # str(), and RAW below: the bonus code is an identifier,
                     # not a quantity.
                     str(row['bonus']),
                     str(row['eventtype']),
                     str(row['attribute']),
                     _num(row['Total_Quantity']),
                     _num(row['Total_StandardHours']),
                     _num(row['Total_SMV']),
                     int(m['week_val']),
                     m['date_time_range'],
                     str(row['report_name'])],
        })
    data_rows.sort(key=lambda d: d['sort'])

empty_blocks = [(ep, m) for ep, m in blk_by_epoch.items() if ep not in blocks_with_data]

if output_target == "Google Sheet":
    # One placeholder per EMPTY block so the sheet marks it covered (matches the
    # live job). One per block is enough; the rebuild cell skips them (Hour == '').
    rows = list(data_rows)
    for ep, m in empty_blocks:
        rows.append({"sort": (ep, 999, 0, ''),
                     "vals": [str(m['date_display']), '', '', '(no data this window)', '', '', '', '',
                              int(m['week_val']), m['date_time_range'], '(no data this window)']})
    rows.sort(key=lambda d: d['sort'])
    values = [d['vals'] for d in rows]
    if pdf is not None and values:
        try:
            ws.append_rows(values, value_input_option='RAW', table_range='A1')
            total_rows_pushed = len(values)
            print(f"Appended {total_rows_pushed} rows to the 'Data' tab in one batch.")

            # --- Record these blocks as covered -------------------------------
            # The live job no longer decides what has been fetched by looking at
            # the Data tab - Archive.js and dailyDataCleanup both prune it, so
            # rows disappearing from Data does not mean the window was missed.
            # It reads the 'Pipeline State' tab instead.
            #
            # Which means a backfill MUST record what it wrote. Without this the
            # live gap-scan would see these blocks as never fetched and pull
            # them again, laying duplicate rows straight on top of the backfill
            # this notebook just performed.
            try:
                _state_header = ["Window End (UTC)", "Date Time Range", "Rows Written", "Recorded At (UTC)"]
                try:
                    _state_ws = sh.worksheet("Pipeline State")
                except gspread.exceptions.WorksheetNotFound:
                    _state_ws = sh.add_worksheet(title="Pipeline State", rows=1000, cols=len(_state_header))
                    _state_ws.update('A1', [_state_header])

                _rows_per_block = {}
                for d in rows:
                    _rows_per_block[d['sort'][0]] = _rows_per_block.get(d['sort'][0], 0) + 1

                _stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
                _end_by_epoch = {_blk_epoch(b['start_utc']): b['end_utc'] for b in blocks}
                _state_ws.append_rows(
                    [[_end_by_epoch[ep], blk_by_epoch[ep]['date_time_range'],
                      _rows_per_block.get(ep, 0), _stamp] for ep in sorted(blk_by_epoch)],
                    value_input_option='RAW', table_range='A1')
                print(f"Recorded {len(blk_by_epoch)} block(s) in 'Pipeline State'.")
            except Exception as e:
                # Not fatal: the rows ARE on the sheet, which is what the user
                # asked for. But say so loudly - an unrecorded backfill will be
                # duplicated by the next live run.
                print(f"WARNING - backfill written but NOT recorded in 'Pipeline State': "
                      f"{type(e).__name__}: {e}")
                print("         The live job may re-pull these blocks and duplicate them.")
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)}"
            print(f"BATCH APPEND FAILED - {error_msg}")
            failures.append({"block": f"{len(blocks)} block(s)",
                             "report": "(sheet append)", "error": error_msg})
else:
    # Databricks-only (default): show the result as a table, write nothing to the sheet.
    result_pdf = pd.DataFrame([d['vals'] for d in data_rows], columns=COLUMNS)
    print("Output target = Databricks table (nothing written to Google Sheet).")
    print(f"{len(result_pdf)} data rows across {len(blocks) - len(empty_blocks)} block(s) with data.")
    if empty_blocks:
        print(f"{len(empty_blocks)} block(s) with no data:")
        for _, m in sorted(empty_blocks, key=lambda t: t[0]):
            print(f"  {m['date_time_range']}")
    display(result_pdf)

# --- Per-block x report breakdown for the run log. ---
if pdf is not None:
    counts = pdf.groupby(['window_epoch', 'report_name']).size()
    for ep in sorted(blk_by_epoch):
        m = blk_by_epoch[ep]
        print(f"\n=== Block: {m['date_time_range']} ===")
        any_rows = False
        for r in reports:
            n = int(counts.get((ep, r['name']), 0))
            if n:
                any_rows = True
                print(f"  [{r['name']}] {n} rows")
        if not any_rows:
            print("  (no data this block)")

print(f"\n--- Backfill summary ---")
print(f"Output target: {output_target}")
print(f"Blocks processed: {len(blocks)}")
print(f"Total rows pushed to sheet: {total_rows_pushed}")
print(f"Failures: {len(failures)}")
for f in failures:
    print(f"  {f['block']} | {f['report']} | {f['error']}")


In [0]:
# ============================================================
# REBUILD "Processed Data (15mins)" from the Data tab
# Only runs when output_target == "Google Sheet" (i.e. this backfill actually
# wrote rows to the Data tab). In "Databricks table" mode nothing was written to
# the sheet, so there is nothing to rebuild and this cell is skipped.
#
# Runs once after ALL blocks are appended above, so the rebuild reflects the
# full backfilled range. Pivot: group raw Data rows by (Date Time Range, Hour, Bonus).
#   D:L = sum(StandardHours) per area / divisor ; M = total ;
#   N:V = sum(Quantity) per area for that area's volume event(s).
# Databricks is the sole writer of this tab (Apps Script no longer processes).
# ============================================================
if output_target != "Google Sheet":
    print("Output target = Databricks table; skipping 'Processed Data (15mins)' sheet rebuild.")
else:
    from datetime import datetime
    from collections import OrderedDict

    def _f(x):
        try:
            return float(str(x).replace(',', '').strip())
        except Exception:
            return 0.0

    def _parse_start(dtr):
        try:
            return datetime.strptime(dtr.split(' - ')[0].strip(), '%d/%m/%Y %H:%M')
        except Exception:
            return datetime.max

    area_index = {a["report"]: i for i, a in enumerate(PROC_AREAS)}
    n_areas = len(PROC_AREAS)

    all_data = ws.get_all_values()            # ws = Data worksheet (from earlier cell)
    data_rows = all_data[1:] if all_data else []

    # Data cols: A=Date B=Hour C=Bonus D=EventType E=Attribute F=Qty G=StdHours
#            H=SMV I=Week J=DateTimeRange K=ReportName
    groups = OrderedDict()
    for r in data_rows:
        if len(r) < 11:
            continue
        hour = r[1].strip()
        if hour == '':                        # skip "(no data this window)" placeholders
            continue
        ai = area_index.get(r[10].strip())    # Report Name
        if ai is None:
            continue
        # .upper() as well as .strip() - see the live notebook: this pivot
        # reads the whole Data tab, including rows that predate the SQL
        # canonicalisation, so case is folded here too.
        key = (r[9].strip(), hour, r[2].strip().upper())   # (Date Time Range, Hour, Bonus)
        g = groups.get(key)
        if g is None:
            g = {"std": [0.0] * n_areas, "vol": [0.0] * n_areas}
            groups[key] = g
        g["std"][ai] += _f(r[6])              # Total_StandardHours (all event types)
        if r[3].strip() in PROC_AREAS[ai]["vol_events"]:
            g["vol"][ai] += _f(r[5])          # Total_Quantity (volume event only)

    out = []
    for (dtr, hour, bonus), g in groups.items():
        std = [s / PROC_DIVISORS[i] for i, s in enumerate(g["std"])]
        try:
            hour_out = int(hour)
        except ValueError:
            hour_out = hour
        row = [dtr, hour_out, bonus] + std + [sum(std)] + g["vol"]
        out.append((_parse_start(dtr), bonus, row))

    out.sort(key=lambda t: (t[0], t[1]))
    proc_values = [t[2] for t in out]

    proc_ws = sh.worksheet("Processed Data (15mins)")

    # Write first, then clear only what is left dangling past the new end.
    #
    # Clearing first left the tab EMPTY for the whole round trip of the update
    # call, and the dashboard polls every minute - so it could read a blank
    # sheet and show a blank dashboard. If the update then failed, the tab
    # stayed empty until something else rebuilt it. Writing over the top means
    # the tab always holds a complete set of rows: the previous one, or this
    # one, never neither.
    #
    # RAW for the same reason as the Data append - column C here is the bonus
    # code, and USER_ENTERED mangled it identically.
    old_last_row = len(proc_ws.col_values(1))      # includes the header row
    new_last_row = len(proc_values) + 1

    # Adding a work area widens the tab. A sheet whose grid is still the old
    # width rejects the wider write outright ("exceeds grid limits"), so grow
    # it first; this is a no-op on every run that does not add a column.
    if proc_ws.col_count < len(PROC_HEADER):
        proc_ws.resize(cols=len(PROC_HEADER))

    # Header and rows go in ONE call, from A1. Written separately there is a
    # window where a new header sits over old rows (or the reverse) - and
    # since the dashboard maps columns by header NAME, that window mis-reads
    # every area after the one that moved rather than failing visibly.
    proc_ws.update("A1", [PROC_HEADER] + proc_values, value_input_option="RAW")

    if old_last_row > new_last_row:
        proc_ws.batch_clear([f"A{new_last_row + 1}:{PROC_LAST_COL}{old_last_row}"])

    print(f"Processed Data (15mins) rebuilt: {len(proc_values)} rows")
